In [1]:
import os, sys, cv2, numpy as np
sys.path.insert(0, os.path.abspath(os.path.join('.', '..', '..', '..', 'src')))

os.environ['OMERO_HOST'] = '100.125.247.59'
os.environ['OMERO_PORT'] = '4064'
os.environ['OMERO_USERNAME'] = 'root'
os.environ['OMERO_PASSWORD'] = 'omero'

from image_container import Image_Container

PROJECT_NAME = 'GR08-6'
COLLECTION_NAME = 'collection_2026-03-02_11-30-59'
OUTPUT_DIR = os.path.join('.', 'Dataset', 'GR08-6', COLLECTION_NAME)

def sanitize(name):
    return name.replace('/', '_').replace('\\\\', '_').replace(':', '_')

print(f'Connecting to OMERO...')
ic = Image_Container()
conn = ic.conn
print('Connected.')

2026-03-08 23:33:01,679 DEBUG [              omero.util.TempFileManager] (MainThread) Added file /Users/austinwu/omero/tmp/.lock_testdu7ko5rv.tmp
2026-03-08 23:33:01,680 DEBUG [              omero.util.TempFileManager] (MainThread) Chose global tmpdir: /Users/austinwu/omero/tmp
2026-03-08 23:33:01,681 DEBUG [              omero.util.TempFileManager] (MainThread) Using temp dir: /Users/austinwu/omero/tmp/omero_austinwu/10717


Connecting to OMERO...
Connected.


In [2]:
project = None
for p in conn.getObjects('Project'):
    if p.getName() == PROJECT_NAME:
        project = p
        break

datasets = list(project.listChildren())
target_ds = None
for ds in datasets:
    if ds.getName() == COLLECTION_NAME:
        target_ds = ds
        break

os.makedirs(OUTPUT_DIR, exist_ok=True)
images = list(target_ds.listChildren())
print(f'{len(images)} total images')

skipped = 0
downloaded = 0
errors = 0
for i, img in enumerate(images):
    img_name = sanitize(img.getName())
    if not img_name.lower().endswith(('.png', '.jpg', '.jpeg', '.tif', '.tiff')):
        img_name += '.png'
    out_path = os.path.join(OUTPUT_DIR, img_name)
    if os.path.exists(out_path):
        skipped += 1
        continue
    print(f'  [{i+1}/{len(images)}] Downloading: {img_name} (id={img.getId()})...', end='', flush=True)
    try:
        arr = ic.download_image(img.getId())
        cv2.imwrite(out_path, arr)
        downloaded += 1
        print(' OK')
    except Exception as e:
        errors += 1
        print(f' ERROR: {e}')

print(f'Done. Skipped {skipped}, downloaded {downloaded}, errors {errors} (total {len(images)})')
ic.disconnect_from_omero()

3800 total images
Done. Skipped 3800, downloaded 0, errors 0 (total 3800)
